# Bundle Authoring Guide Notebook

Mirrors the [Bundle Authoring Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_authoring_guide/). Each section follows the same headings as the guide.

Run from `docs/assets/` so the relative path to `MODELS/POST82.yaml` resolves.

In [1]:
import io

import numpy as np
import pandas as pd

from scipy.stats import laplace

from SymbolicDSGE import BundleBuilder, Estimator, Shock, parse_config, solve_model
from SymbolicDSGE.estimation import make_prior

from SymbolicDSGE.monte_carlo import MCPipeline
from SymbolicDSGE.monte_carlo.step_factories import (
    jarque_bera_test_step,
    simulation_step,
)

## Solve a reference model

In [2]:
# Create the Bundle
bundle = BundleBuilder(created_by="experiment-1")
config = parse_config("../../MODELS/POST82.yaml")

sol = solve_model(
    config,
    ss_seed=[0.0, 0.0, 0.0, 0.0, 0.0],
)

# Add a model to the bundle
bundle.add_model(
    "reference",
    config,
    solve_kwargs={"ss_seed": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

# The MC experiment uses a second model. add_mc(...) does not bind
# models to a bundle, we must register all models referenced in a member.
# You can register the model after the MC pipeline, presence is required
# at build/write time.
bundle.add_model(
    "dgp",
    config,  # same model as "reference" here, but could be any model
    solve_kwargs={"ss_seed": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

## Specify the estimation tab

This section defines a small MCMC run against synthetic observed data, then stores the live `Estimator` and `MCMCResult` in the bundle.

In [3]:
estimated = {
    "psi_pi": make_prior(
        distribution="normal",
        parameters={"mean": 1.5, "std": 0.25},
        transform="identity",
    ),
    "psi_x": make_prior(
        distribution="normal",
        parameters={"mean": 0.5, "std": 0.2},
        transform="identity",
    ),
}

rng = np.random.default_rng(0)
observed = rng.standard_normal((40, 3))
estim = Estimator(
    config=config,
    observables=["OutGap", "Infl", "Rate"],
    y=observed,
    estimated_params=estimated,
)
res = estim.mcmc(n_draws=1000, burn_in=200, thin=2, random_state=0)

MCMC sampling concluded in 0.14 seconds with 15459.42 iterations per second.
[Estimator:mcmc] BK stability warnings encountered during search: [122]


In [4]:
# Add the estimation to the bundle with results
bundle.add_estimation(
    model="reference",
    source=estim,
    result=res,
)

## Build a Monte Carlo pipeline

In [12]:
gz_shock = Shock(seed=0, dist="norm").joint("e_g", "e_z")
r_shock = Shock.from_ppf(laplace, seed=1, loc=1.0).joint("e_r")

mc_pipeline = MCPipeline(
    [
        simulation_step(
            "datagen",
            n_retain=0,
            target="dgp",
            T=200,
            shocks=[gz_shock, r_shock],
        ),
        jarque_bera_test_step(
            "jb_test", n_retain=100, source="datagen", field="observables", column=0
        ),
    ]
)
mc_res = mc_pipeline.run(
    models={
        "reference": sol,
        "dgp": sol,
    },
    n_rep=1000,
    n_jobs=-1,
    verbosity=2,
)

MC run concluded successfully in 0.00s with 534387.86 it/s.
Per-step Report:

	datagen: 0 failures, 53576.50 worker it/s (0.02 worker-s), 534387.86 wall it/s.
	jb_test: 0 failures, 1173295.75 worker it/s (0.00 worker-s), 534387.86 wall it/s.


In [6]:
# Add the Monte Carlo pipeline to the bundle
bundle.add_mc(
    pipeline=mc_pipeline,
    result=mc_res,
)

## Specify a simulation prefill

In [7]:
# Add the simulation prefill to the bundle, keyed by name
bundle.set_simulation(
    "reference",
    T=200,
    observables=True,
    shock_scale=1.0,
    shocks=[
        Shock(seed=42, dist="t", df=3).joint("e_r"),
    ],
)

## Add raw data

In [8]:
# Make text data
aux = pd.DataFrame(
    {
        "date": pd.date_range("2000-01-01", periods=40, freq="QS"),
        "gdp_growth": rng.standard_normal(40),
    }
)
csv_buf = io.StringIO()
aux.to_csv(csv_buf, index=False)

bundle.add_raw_data(
    name="auxiliary_series",
    data=csv_buf.getvalue(),
)

## Write the bundle to disk

In [9]:
bundle.write("experiment-1.sdsge")

WindowsPath('experiment-1.sdsge')

## Inspect the result

In [10]:
!unzip -l experiment-1.sdsge

Archive:  experiment-1.sdsge
  Length      Date    Time    Name
---------  ---------- -----   ----
     3826  22-11-2026 19:31   manifest.json
     1292  22-11-2026 19:31   model/reference.yaml
     1292  22-11-2026 19:31   model/dgp.yaml
      675  22-11-2026 19:31   estimation/spec.json
     1891  22-11-2026 19:31   estimation/observed.parquet
    19600  22-11-2026 19:31   estimation/posterior.parquet
      974  22-11-2026 19:31   estimation/result.json
     7131  22-11-2026 19:31   shock/table/aadb84934fad3b59.parquet
     1286  22-11-2026 19:31   montecarlo/pipeline.json
      535  22-11-2026 19:31   montecarlo/result/meta.json
      538  22-11-2026 19:31   montecarlo/result/datagen/datagen_steps.json
      175  22-11-2026 19:31   montecarlo/result/tests/test_steps.json
     2276  22-11-2026 19:31   montecarlo/result/tests/test_traces.parquet
      305  22-11-2026 19:31   simulation.json
     1282  22-11-2026 19:31   data/auxiliary_series.parquet
---------                     -----

In [11]:
# File size of the bundle in KB
import os

sizeof = round(os.path.getsize("experiment-1.sdsge") / 1024)
print(f"Bundle size: {sizeof} KB")

Bundle size: 37 KB
